# ETL-14 · Texto e encoding

- **Etapa:** `03_texto_encoding` (ETL-14)
- **Responsável:** Guilherme Calistrato (`@guicalistrato`)
- **Revisor:** Davi Mello (`@davi01mello`)
- **Issue:** #14 (ETL-14)
- **Entrada:** `data/intermediarios/01_bruto.parquet` *(na execução final, usar a saída da etapa `02_chaves_duplicatas`)*
- **Saída:** `data/intermediarios/03_texto.parquet`

## 1. Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar o marcador .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

if str(RAIZ / "src") not in sys.path:
    sys.path.append(str(RAIZ / "src"))

from perfil import _p4_texto  # reutiliza o detector de problemas de texto da ETL-06

DADOS  = RAIZ / "data"                      # CSVs brutos (estão no Git)
INTERM = RAIZ / "data" / "intermediarios"   # .parquet (o .gitignore já ignora)
INTERM.mkdir(parents=True, exist_ok=True)

ENTRADA = INTERM / "01_bruto.parquet"       # na execução final, trocar para 02_chaves.parquet
SAIDA   = INTERM / "03_texto.parquet"

# Colunas de identificação, datas, métricas e rastreio que não passam pela limpeza textual
COLUNAS_NAO_TEXTO = {
    "id_sinistro", "id_pessoa", "id_veiculo", "cod_ibge", "idade",
    "data_sinistro", "ano_sinistro", "mes_sinistro", "dia_sinistro", "ano_mes_sinistro",
    "data_obito", "ano_obito", "mes_obito", "dia_obito", "ano_mes_obito",
    "tempo_sinistro_obito", "ano_origem", "arquivo_origem",
}

## 2. Carregar a entrada

In [2]:
df = pd.read_parquet(ENTRADA)
print("shape:", df.shape)
df.head()

shape: (806977, 33)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,sexo,idade,gravidade_lesao,faixa_etaria_demografica,faixa_etaria_legal,profissao,grau_de_instrucao,nacionalidade,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,ano_mes_sinistro,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem
0,2240338,1979797,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,FEMININO,31,ILESO,30 a 34,30-34,EMPRESARIO(A),MÉDIO,BRASILEIRA,01/01/2021,2021,01,01,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,data/pessoas_01-2021.csv
1,2240338,1979798,2874666,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,MASCULINO,34,ILESO,30 a 34,30-34,MOTO-BOY,NaN,BRASILEIRA,01/01/2021,2021,01,01,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,data/pessoas_01-2021.csv
2,1447800,1284984,NaN,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,FEMININO,65,ILESO,65 a 69,65-69,DO LAR,SUPERIOR,BRASILEIRA,01/01/2021,2021,01,01,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,data/pessoas_01-2021.csv
3,1783410,563279,2648403,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,MASCULINO,26,GRAVE,25 a 29,25-29,AUTONOMO(A),NaN,BRASILEIRA,01/01/2021,2021,01,01,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,data/pessoas_01-2021.csv
4,1798856,687322,NaN,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,MASCULINO,66,LEVE,65 a 69,65-69,APOSENTADO(A),SUPERIOR,BRASILEIRA,01/01/2021,2021,01,01,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,data/pessoas_01-2021.csv


## 3. Diagnóstico ANTES

In [3]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "pct_nulo": (df.isna().mean() * 100).round(2),
        "n_distintos": df.nunique(),
    })


def listar_colunas_texto(df: pd.DataFrame) -> list[str]:
    """Seleciona as colunas textuais/categóricas do DataFrame, excluindo IDs, datas e rastreio."""
    return [
        c for c in df.columns
        if c not in COLUNAS_NAO_TEXTO
        and (pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c]))
    ]


antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,str,0.00,402695
id_pessoa,str,0.00,806977
id_veiculo,str,21.04,554294
cod_ibge,str,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


In [4]:
colunas_texto = listar_colunas_texto(df)
det_antes, res_antes = _p4_texto(df[colunas_texto])

print(f"Colunas de texto avaliadas ({len(colunas_texto)}):", colunas_texto)
print("\nResumo de problemas detectados por src/perfil.py (_p4_texto):")
display(res_antes)

print("\nDetalhe dos problemas (excluindo nacionalidade com espaço nas pontas):")
display(det_antes[det_antes["coluna"] != "nacionalidade"])

municipios_sem_apostrofo = (
    df[df["municipio"].str.contains(r"\bD [AO]", regex=True, na=False)]["municipio"]
    .value_counts()
    .rename_axis("municipio")
    .reset_index(name="linhas")
)
print("\nMunicípios com apóstrofo substituído por espaço (D OESTE / D ALHO):")
display(municipios_sem_apostrofo)

Colunas de texto avaliadas (15): ['municipio', 'regiao_administrativa', 'tipo_via', 'tipo_veiculo_vitima', 'tipo_de_vitima', 'modo_transporte_vitima', 'sexo', 'gravidade_lesao', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'profissao', 'grau_de_instrucao', 'nacionalidade', 'local_obito', 'local_via']

Resumo de problemas detectados por src/perfil.py (_p4_texto):


,coluna,motivo,valores_distintos,linhas
0,profissao,espaco_duplo,1,1
1,profissao,letra_isolada,1,1
2,nacionalidade,espaco_nas_pontas,90,711291



Detalhe dos problemas (excluindo nacionalidade com espaço nas pontas):


,coluna,valor,n,motivo
0,profissao,TECNICO EM MECANICA PRODUCAO,1,espaco_duplo
1,profissao,Tecnico em raio x,1,letra_isolada



Municípios com apóstrofo substituído por espaço (D OESTE / D ALHO):


,municipio,linhas
0,SANTA BARBARA D OESTE,4111
1,PALMEIRA D OESTE,231
2,ESTRELA D OESTE,161
3,APARECIDA D OESTE,83
4,SANTA RITA D OESTE,36
5,SAO JOAO DO PAU D ALHO,18
6,SANTA CLARA D OESTE,15
7,GUARANI D OESTE,10


## 4. Transformações

Cada transformação é uma função com docstring. Nenhuma linha deve ser removida nesta etapa.

**Decisões de escopo adotadas:**
1. **Apóstrofo nos municípios (`D OESTE` / `D ALHO`):** Os 8 municípios paulistas que perderam o apóstrofo na fonte (ex.: `SANTA BARBARA D OESTE`, `SAO JOAO DO PAU D ALHO`) são padronizados **com apóstrofo** (`SANTA BARBARA D'OESTE`, `SAO JOAO DO PAU D'ALHO`), mantendo a ausência de acentos alinhada aos demais 637 municípios da coluna `municipio`.
2. **Acentuação e encoding:** Na leitura correta dos CSVs em `latin-1`, nenhum caractere inválido (`U+FFFD`) ou *mojibake* ocorre na base; ainda assim, o `MAPA_TEXTO` inclui entradas preventivas para os casos observados em amostras anteriores (`METROPOLITANA DE S O PAULO`, `M DIO`), garantindo proteção caso o pipeline receba extrações com perda de acento.
3. **Caixa alta/baixa em `profissao` (delegada para a ETL-17):** Foram identificados 266 valores distintos de `profissao` em caixa mista/minúscula (ex.: `'Motorista'`, `'Estudante'`, `'Tecnico em raio x'`) coexistindo com grafias em maiúsculas (`'MOTORISTA'`, `'ESTUDANTE'`). Optou-se por **não converter a caixa de `profissao` na ETL-14**, deixando essa padronização para a **ETL-17 (Padronização de profissão)**, pois a ETL-14 trata exclusivamente da integridade de encoding/acentos, apóstrofos perdidos e espaços excedentes — preservando as grafias originais para a normalização léxica e o agrupamento em `grupo_profissao` na ETL-17. Apenas espaços duplos internos (como `'TECNICO EM  MECANICA PRODUCAO'`) e eventuais espaços nas pontas são removidos aqui.

In [5]:
MAPA_TEXTO: dict[str, dict[str, str]] = {
    "municipio": {
        "SANTA BARBARA D OESTE": "SANTA BARBARA D'OESTE",
        "PALMEIRA D OESTE": "PALMEIRA D'OESTE",
        "ESTRELA D OESTE": "ESTRELA D'OESTE",
        "APARECIDA D OESTE": "APARECIDA D'OESTE",
        "SANTA RITA D OESTE": "SANTA RITA D'OESTE",
        "SAO JOAO DO PAU D ALHO": "SAO JOAO DO PAU D'ALHO",
        "SANTA CLARA D OESTE": "SANTA CLARA D'OESTE",
        "GUARANI D OESTE": "GUARANI D'OESTE",
    },
    # Mapeamento preventivo para acentos eventualmente corrompidos em amostras/extrações
    "regiao_administrativa": {
        "METROPOLITANA DE S O PAULO": "METROPOLITANA DE SÃO PAULO",
        "METROPOLITANA DE SAO PAULO": "METROPOLITANA DE SÃO PAULO",
        "S O JOSÉ DOS CAMPOS": "SÃO JOSÉ DOS CAMPOS",
        "S O JOS DOS CAMPOS": "SÃO JOSÉ DOS CAMPOS",
        "S O JOSÉ DO RIO PRETO": "SÃO JOSÉ DO RIO PRETO",
        "S O JOS DO RIO PRETO": "SÃO JOSÉ DO RIO PRETO",
        "RIBEIR O PRETO": "RIBEIRÃO PRETO",
        "MAR LIA": "MARÍLIA",
        "ARA ATUBA": "ARAÇATUBA",
    },
    "grau_de_instrucao": {
        "M DIO": "MÉDIO",
        "B SICO": "BÁSICO",
    },
}


def normalizar_espacos(df: pd.DataFrame, colunas: list[str] | None = None) -> pd.DataFrame:
    """Remove espaços nas pontas e colapsa espaços internos múltiplos nas colunas de texto."""
    df = df.copy()
    alvo = colunas if colunas is not None else listar_colunas_texto(df)
    for col in alvo:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    return df


def corrigir_encoding(
    df: pd.DataFrame,
    mapa: dict[str, dict[str, str]] | None = None,
) -> pd.DataFrame:
    """Aplica o mapa explícito {valor_errado: valor_certo} nas colunas de texto informadas."""
    df = df.copy()
    regras = mapa if mapa is not None else MAPA_TEXTO
    for col, substituicoes in regras.items():
        if col in df.columns and substituicoes:
            df[col] = df[col].replace(substituicoes)
    return df


df_antes = df
n_antes = len(df_antes)

df = normalizar_espacos(df_antes, colunas_texto)
df = corrigir_encoding(df, MAPA_TEXTO)

print(f"linhas: {n_antes} -> {len(df)} (removidas: {n_antes - len(df)})")

auditoria_colunas = []
for col in colunas_texto:
    s_antes = df_antes[col]
    s_depois = df[col]
    alteradas = int((s_antes.fillna("<NA>") != s_depois.fillna("<NA>")).sum())
    if alteradas > 0 or s_antes.nunique() != s_depois.nunique():
        auditoria_colunas.append({
            "coluna": col,
            "celulas_alteradas": alteradas,
            "distintos_antes": int(s_antes.nunique()),
            "distintos_depois": int(s_depois.nunique()),
        })

print("\nResumo das alterações por coluna:")
display(pd.DataFrame(auditoria_colunas))

auditoria_mapa = []
for col, regras in MAPA_TEXTO.items():
    if col not in df_antes.columns:
        continue
    for errado, certo in regras.items():
        qtd = int((df_antes[col] == errado).sum())
        if qtd > 0:
            auditoria_mapa.append({
                "coluna": col,
                "valor_errado": errado,
                "valor_certo": certo,
                "linhas_corrigidas": qtd,
            })

print("\nOcorrências efetivamente corrigidas pelo MAPA_TEXTO:")
display(pd.DataFrame(auditoria_mapa))

linhas: 806977 -> 806977 (removidas: 0)

Resumo das alterações por coluna:


,coluna,celulas_alteradas,distintos_antes,distintos_depois
0,municipio,4665,645,645
1,profissao,1,814,814
2,nacionalidade,711291,90,90



Ocorrências efetivamente corrigidas pelo MAPA_TEXTO:


,coluna,valor_errado,valor_certo,linhas_corrigidas
0,municipio,SANTA BARBARA D OESTE,SANTA BARBARA D'OESTE,4111
1,municipio,PALMEIRA D OESTE,PALMEIRA D'OESTE,231
2,municipio,ESTRELA D OESTE,ESTRELA D'OESTE,161
3,municipio,APARECIDA D OESTE,APARECIDA D'OESTE,83
4,municipio,SANTA RITA D OESTE,SANTA RITA D'OESTE,36
5,municipio,SAO JOAO DO PAU D ALHO,SAO JOAO DO PAU D'ALHO,18
6,municipio,SANTA CLARA D OESTE,SANTA CLARA D'OESTE,15
7,municipio,GUARANI D OESTE,GUARANI D'OESTE,10


## 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [6]:
depois = diagnostico(df)
comparacao = antes.join(depois, lsuffix="_antes", rsuffix="_depois")
comparacao

,dtype_antes,pct_nulo_antes,n_distintos_antes,dtype_depois,pct_nulo_depois,n_distintos_depois
id_sinistro,str,0.00,402695,str,0.00,402695
id_pessoa,str,0.00,806977,str,0.00,806977
id_veiculo,str,21.04,554294,str,21.04,554294
cod_ibge,str,0.00,645,str,0.00,645
municipio,str,0.00,645,str,0.00,645
regiao_administrativa,str,0.00,16,str,0.00,16
tipo_via,str,0.00,3,str,0.00,3
tipo_veiculo_vitima,str,20.98,7,str,20.98,7
tipo_de_vitima,str,0.00,4,str,0.00,4
modo_transporte_vitima,str,20.54,8,str,20.54,8


In [7]:
det_depois, res_depois = _p4_texto(df[colunas_texto])
print("Problemas de texto restantes (_p4_texto):")
display(res_depois)
display(det_depois)

# Conferência da integridade 1:1 de cod_ibge -> municipio (pré-requisito da ETL-21)
mun_por_ibge = df.groupby("cod_ibge")["municipio"].nunique()
inconsistentes_ibge = mun_por_ibge[mun_por_ibge > 1]
print(f"Códigos IBGE distintos: {len(mun_por_ibge)} | Municípios distintos: {df['municipio'].nunique()}")
print(f"Códigos IBGE com mais de um município: {len(inconsistentes_ibge)}")
assert len(inconsistentes_ibge) == 0, "Há cod_ibge associado a mais de um município!"
assert len(df) == n_antes, "Nenhuma linha deve ser removida na ETL-14!"
assert (df_antes.isna().sum() == df.isna().sum()).all(), "Contagem de nulos alterada indevidamente!" 

Problemas de texto restantes (_p4_texto):


,coluna,motivo,valores_distintos,linhas
0,profissao,letra_isolada,1,1


,coluna,valor,n,motivo
0,profissao,Tecnico em raio x,1,letra_isolada


Códigos IBGE distintos: 645 | Municípios distintos: 645
Códigos IBGE com mais de um município: 0


## 6. Salvar a saída

In [8]:
df.to_parquet(SAIDA, index=False)
print("salvo em:", SAIDA, "| shape:", df.shape)

salvo em: C:\Users\gcnt1\.gemini\antigravity\worktrees\Integracao_BD\etl14_planejamento\data\intermediarios\03_texto.parquet | shape: (806977, 33)


## 7. Resumo

**O que foi feito:** Limpeza de texto, verificação de encoding e padronização ortográfica de municípios em todas as 15 colunas textuais da base unificada (`806.977` linhas × `33` colunas), mantendo `0` linhas removidas e `0` alterações na contagem de nulos.
- `normalizar_espacos()`: removeu espaços nas pontas e colapsou espaços internos múltiplos. Corrigiu **711.291** registros de `nacionalidade` (todos os 90 valores distintos vinham preenchidos com espaços à direita) e **1** registro de `profissao` com espaço duplo (`'TECNICO EM  MECANICA PRODUCAO'` → `'TECNICO EM MECANICA PRODUCAO'`).
- `corrigir_encoding()`: aplicou o `MAPA_TEXTO` (`{valor_errado: valor_certo}`) e restaurou o apóstrofo em **8 municípios** (`4.665` linhas no total: `SANTA BARBARA D'OESTE`, `PALMEIRA D'OESTE`, `ESTRELA D'OESTE`, `APARECIDA D'OESTE`, `SANTA RITA D'OESTE`, `SAO JOAO DO PAU D'ALHO`, `SANTA CLARA D'OESTE`, `GUARANI D'OESTE`), preservando a relação 1:1 com os 645 códigos IBGE. Também manteve entradas preventivas no mapa para acentos que apareceram corrompidos apenas em amostras antigas (`METROPOLITANA DE S O PAULO`, `M DIO`), confirmando `0` ocorrências corrompidas na leitura em `latin-1`.

**Por quê:** Espaços excedentes e grafias divergentes quebram agrupamentos, `JOIN`s e a criação das dimensões `dim_localidade` (`ETL-21`) e `dim_profissao` (`ETL-17` / `ETL-22`). A conversão de caixa de `profissao` (266 valores em caixa mista/minúscula) foi intencionalmente deixada para a `ETL-17`, onde será tratada junto com a unificação léxica e o agrupamento em `grupo_profissao`.

**Resultado:** **715.957** células corrigidas sem perda de registros; no diagnóstico final com `_p4_texto`, resta apenas 1 falso positivo conhecido (`'Tecnico em raio x'`, 1 linha, sinalizado por conter a letra `'x'`).